# Adım 6 — Karşılaştırma deneyleri (tek notebook, ayar hücresinden yönetilir)

Baseline notebook'u (`train_baseline.ipynb`) olduğu gibi duruyor. Bu notebook **her deney için aynı**; değişen yalnızca
**Bölüm 4'teki ayar hücresi**. Ortak ayarlar baseline ile aynı: batch 16, patience 10, save_period 5, seed 42,
cache kapalı, workers 4, Ultralytics varsayılan augmentation.

**Kaggle ayarları (sağ panel):** Accelerator = **GPU T4**, Internet = **açık**, "Add Input" ile `rdd2022_yolo`.

| # | Deney | MODEL | IMGSZ | NAME | EXTRA | OVERSAMPLE_D40 |
|---|---|---|---|---|---|---|
| 2 | YOLO11s | `yolo11s.pt` | 640 | `e2_yolo11s` | `{}` | 1 |
| 3 | YOLO26s | `yolo26s.pt` | 640 | `e3_yolo26s` | `{}` | 1 |
| 4 | YOLO26n | `yolo26n.pt` | 640 | `e4_yolo26n` | `{}` | 1 |
| 5 | Seçilen model, 1024 | seçilen | 1024 | `e5_<model>_1024` | `{}` | 1 |
| 6 | D40: sınıf ağırlığı | seçilen | 640 | `e6_<model>_clspw` | `{"cls_pw": 0.5}` | 1 |
| 7 | D40: oversampling | seçilen | 640 | `e7_<model>_d40x2` | `{}` | 2 |

**MODE ne zaman ne olur?**
1. `"deneme"`: yeni bir model/imgsz ilk kez denenecekse. Train'in %10'u, 2 epoch. 1. epoch sonunda süre sınırı
   bilerek tetiklenir, aynı hücre hemen `last.pt`'den devam ettirir. Model indirme, bellek (batch) ve dur+devam test edilir.
2. `"egitim"`: asıl eğitim. **Save Version → Save & Run All (Commit)**. Tarayıcı kapansa da çalışır.
3. `"devam"`: eğitim süre sınırıyla durduysa (log'da `SÜRE SINIRI`), yeni oturumda önceki versiyonun output'unu
   "Add Input" ile ekle, `PREV_OUTPUT` = o output'taki `runs/<NAME>` yolu → Save & Run All.

Eğitim bitince output'taki **`<NAME>_indir.zip`** indirilir (yerel adımlar: `docs/adimlar/adim-6-karsilastirma.md`).

## 1. GPU ve kurulum

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.4.182
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

## 2. Veriyi bul ve kontrol et
Baseline ile aynı: `data.yaml` aranarak bulunur, sayılar Adım 3 ile tutmalı (21.211 / 3.838 / 3.838).

In [ ]:
from pathlib import Path

EXPECTED = {"train": 21211, "val": 3838, "test": 3838}

# Kaggle zip'i açarken klasör adını kendisi koyuyor; bu yüzden data.yaml'ı arayarak buluyoruz
candidates = [p.parent for p in Path("/kaggle/input").glob("**/data.yaml") if (p.parent / "train.txt").exists()]
assert len(candidates) == 1, f"Tek veri klasörü bekleniyordu, bulunan: {candidates}"
DATA_DIR = candidates[0]
print("Veri klasörü:", DATA_DIR)

# Liste, görüntü ve etiket sayıları birbirini ve Adım 3'ü tutmalı (eksik yükleme olursa burada durur)
for part, n in EXPECTED.items():
    lines = (DATA_DIR / f"{part}.txt").read_text(encoding="utf-8").splitlines()
    n_img = sum(1 for _ in (DATA_DIR / "images" / part).iterdir())
    n_lbl = sum(1 for _ in (DATA_DIR / "labels" / part).iterdir())
    first = DATA_DIR / lines[0]
    print(f"{part}: liste {len(lines)}, görüntü {n_img}, etiket {n_lbl}, ilk satır var mı: {first.exists()}")
    assert len(lines) == n_img == n_lbl == n and first.exists()

## 3. data.yaml'ı /kaggle/working'e yaz
`/kaggle/input` salt okunur. Yeni yaml `path` ile input klasörünü gösterir. **Devamdan önce de çalışmalı:**
checkpoint bu yaml'ın yolunu saklıyor.

In [ ]:
NAMES = "names:\n  0: D00\n  1: D10\n  2: D20\n  3: D40\n"  # sınıf sırası eğitimdekiyle birebir aynı olmalı

DATA_YAML = Path("/kaggle/working/data.yaml")
DATA_YAML.write_text(f"path: {DATA_DIR}\ntrain: train.txt\nval: val.txt\ntest: test.txt\n\n" + NAMES, encoding="utf-8")
print(DATA_YAML.read_text())

## 4. AYAR HÜCRESİ — her deneyde yalnızca burası değişir
- `MODEL`: COCO ön-eğitimli ağırlık (Ultralytics ilk kullanımda indirir).
- `EXTRA`: tek değişkenli deneyin ek ayarı, ör. `{"cls_pw": 0.5}` (sınıf ağırlığı). Boşsa baseline ile aynı.
- `OVERSAMPLE_D40`: 2 → D40 (çukur) içeren train görüntüleri listede 2 kez geçer.
- `MAX_HOURS`: oturum içi süre sınırı (saat). Bir sonraki epoch bu sınırı aşacaksa eğitim, `last.pt` kaydedildikten
  sonra düzgünce durur ve `"devam"` ile sürdürülebilir. Kaggle'ın 12 saatte zorla kesmesine kalmaz.

In [ ]:
MODE = "deneme"          # "deneme" | "egitim" | "devam"

MODEL = "yolo11s.pt"     # yolo11s.pt | yolo26s.pt | yolo26n.pt | seçilen model
IMGSZ = 640              # 640; 1024 deneyinde 1024
EPOCHS = 50
NAME = "e2_yolo11s"      # çıktı klasörü: /kaggle/working/runs/<NAME>
EXTRA = {}               # ör. {"cls_pw": 0.5}
OVERSAMPLE_D40 = 1       # 1 = yok, 2 = D40'lı görüntüler 2 kez

MAX_HOURS = 11           # oturum içi süre sınırı (Kaggle sınırı 12 sa)
PREV_OUTPUT = None       # "devam" için: ör. "/kaggle/input/<notebook>/runs/e5_yolo11s_1024"

# ---- Ortak ayarlar: baseline ile aynı, DEĞİŞTİRME (karşılaştırmada tek değişken değişmeli) ----
RUNS = Path("/kaggle/working/runs")   # tam yol: göreli verilirse Ultralytics başına runs/detect ekliyor
COMMON = dict(
    imgsz=IMGSZ, seed=42, cache=False, batch=16, workers=4,
    patience=10, save_period=5, project=str(RUNS), device=0,
)
print(f"MODE={MODE} | {MODEL} | imgsz {IMGSZ} | {EPOCHS} epoch | {NAME} | EXTRA={EXTRA} | D40x{OVERSAMPLE_D40} | sınır {MAX_HOURS} sa")

## 5. (Yalnızca OVERSAMPLE_D40 > 1) D40 oversampling listesi
Etiket dosyasında sınıf 3 (D40) geçen train görüntüleri bulunur. Yeni liste = tüm train satırları + bu görüntüler
(`OVERSAMPLE_D40 − 1`) kez daha. Görüntüler kopyalanmaz, sadece listede tekrar eder; Ultralytics tekrar eden satırları
ayrı örnek sayar. Val ve test değişmez. **Devamdan önce de çalışmalı** (checkpoint yaml yolunu saklıyor).

In [ ]:
TRAIN_YAML = DATA_YAML  # varsayılan: baseline ile aynı veri

if OVERSAMPLE_D40 > 1:
    lines = (DATA_DIR / "train.txt").read_text(encoding="utf-8").splitlines()
    full, d40 = [], []
    for line in lines:
        img = DATA_DIR / line                                            # "./images/train/X.jpg" → mutlak yol
        lbl = DATA_DIR / "labels" / "train" / (Path(line).stem + ".txt")
        full.append(str(img))                                            # mutlak yol: liste /kaggle/working'de duracak
        rows = lbl.read_text(encoding="utf-8").splitlines() if lbl.exists() else []
        if any(r.split()[0] == "3" for r in rows if r.strip()):          # satırın ilk sayısı sınıf numarası
            d40.append(str(img))
    new_list = full + d40 * (OVERSAMPLE_D40 - 1)
    list_path = Path(f"/kaggle/working/train_d40x{OVERSAMPLE_D40}.txt")
    list_path.write_text("\n".join(new_list) + "\n", encoding="utf-8")
    TRAIN_YAML = Path(f"/kaggle/working/data_d40x{OVERSAMPLE_D40}.yaml")
    TRAIN_YAML.write_text(
        f"path: {DATA_DIR}\ntrain: {list_path}\nval: val.txt\ntest: test.txt\n\n" + NAMES, encoding="utf-8")
    print(f"Train {len(full)} görüntü, D40 içeren {len(d40)} → yeni liste {len(new_list)} satır")

print("Eğitimde kullanılacak yaml:", TRAIN_YAML)

## 6. Süre sınırı (callback)
Ultralytics her epoch sonunda önce `last.pt`'yi kaydeder, sonra `on_fit_epoch_end` callback'lerini çağırır.
Buraya eklenen fonksiyon "geçen süre + bir epoch daha > MAX_HOURS" ise `SureDoldu` hatası fırlatır. Eğitim
`final_eval`'e girmeden durduğu için `last.pt` optimizer'ı ile kalır → `resume=True` ile devam edilebilir.
Son epoch'ta veya erken durmada (`trainer.stop`) dokunmaz; eğitim normal biter.

In [ ]:
import time


class SureDoldu(Exception):
    """Süre sınırı yüzünden bilerek durdurulduğunu belirten hata (gerçek bir hata değil)."""


def sure_siniri_ekle(model, max_hours):
    """Modele, max_hours dolmadan bir sonraki epoch'a girmeyi engelleyen callback ekler."""
    start = time.time()  # sayaç bu hücre çalışınca başlar (kurulum süresi zaten harcandı, pay bırakıyoruz)

    def kontrol(trainer):
        if trainer.stop:  # son epoch ya da erken durma: normal bitişe izin ver
            return
        gecen = time.time() - start
        if gecen + trainer.epoch_time > max_hours * 3600:  # bir epoch daha sığmıyor
            raise SureDoldu(f"epoch {trainer.epoch + 1} sonunda durduruldu, geçen {gecen / 3600:.2f} sa")

    model.add_callback("on_fit_epoch_end", kontrol)
    return model


def egit(model, max_hours, **kwargs):
    """model.train'i süre sınırıyla çalıştırır. Dönüş: (koşu klasörü, eğitim bitti mi?)."""
    sure_siniri_ekle(model, max_hours)
    try:
        model.train(**kwargs)
        bitti = True
    except SureDoldu as e:
        print(f"\nSÜRE SINIRI: {e}. last.pt hazır; yeni oturumda MODE='devam' ile sürdür.")
        bitti = False
    save_dir = Path(model.trainer.save_dir)
    if model.trainer.batch_size != 16:  # 8.4.182 ilk epoch'ta bellek yetmezse batch'i kendiliğinden yarıya indiriyor
        print(f"UYARI: gerçek batch {model.trainer.batch_size} (16 değil) → deneyler.md'ye not edilmeli")
    return save_dir, bitti


def devam_et(run_dir, max_hours):
    """run_dir/weights/last.pt'den kaldığı epoch'tan devam eder (ayarlar checkpoint'ten okunur)."""
    last = run_dir / "weights" / "last.pt"
    assert last.exists(), f"{last} yok"
    return egit(YOLO(str(last)), max_hours, resume=True)

## 7. Eğitim (MODE'a göre tek hücre)
- `"deneme"`: train'in %10'u (`fraction=0.1`), 2 epoch, süre sınırı 0 → 1. epoch sonunda durur, hemen devam eder.
  Log'da önce `SÜRE SINIRI`, sonra `Resuming training ... from epoch 2` görülmeli.
- `"egitim"`: asıl eğitim. - `"devam"`: `PREV_OUTPUT` aynı yola kopyalanır (checkpoint mutlak yolları saklıyor).

In [ ]:
from ultralytics import YOLO
import shutil

if MODE == "deneme":
    run_dir, bitti = egit(YOLO(MODEL), 0, data=str(TRAIN_YAML), **COMMON, **EXTRA,
                          epochs=2, fraction=0.1, name=f"deneme_{NAME}")
    assert not bitti, "Süre sınırı tetiklenmedi"
    run_dir, bitti = devam_et(run_dir, MAX_HOURS)       # aynı oturumda resume testi
    assert bitti, "Devam koşusu bitmedi"
    print("DENEME BAŞARILI:", run_dir)

elif MODE == "egitim":
    run_dir, bitti = egit(YOLO(MODEL), MAX_HOURS, data=str(TRAIN_YAML), **COMMON, **EXTRA,
                          epochs=EPOCHS, name=NAME)

elif MODE == "devam":
    run_dir = RUNS / NAME
    if not run_dir.exists():
        assert PREV_OUTPUT, "PREV_OUTPUT verilmeli (önceki versiyonun output'undaki runs/<NAME>)"
        shutil.copytree(PREV_OUTPUT, run_dir)            # aynı yola: checkpoint /kaggle/working/runs/<NAME> bekliyor
    run_dir, bitti = devam_et(run_dir, MAX_HOURS)

else:
    raise ValueError(f"Bilinmeyen MODE: {MODE}")

print("Koşu klasörü:", run_dir, "| bitti mi:", bitti)

## 8. Sonuç özeti ve indirme zip'i
Eğitim bittiyse `<NAME>_indir.zip` oluşturulur: `best.pt` + ayarlar + grafikler. `last.pt` ve `epoch*.pt` zip'e
girmez (büyük; output'ta duruyorlar). Süre sınırıyla durduysa zip yapılmaz, sıradaki oturumda `"devam"`.

In [ ]:
import pandas as pd
import zipfile

csv = run_dir / "results.csv"
if csv.exists():
    df = pd.read_csv(csv)
    df.columns = df.columns.str.strip()
    print(f"== {run_dir.name}: {len(df)} epoch, en iyi val mAP50 {df['metrics/mAP50(B)'].max():.3f} "
          f"(Kaggle eğitim sonu değeri; resmî değer yerelde evaluate.py ile)")
    print(df[["epoch", "train/box_loss", "train/cls_loss", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(5).to_string(index=False))

if bitti:
    zip_path = Path(f"/kaggle/working/{run_dir.name}_indir.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(run_dir.rglob("*")):
            if f.is_file() and (f.name == "best.pt" or f.suffix in {".yaml", ".csv", ".png", ".jpg"}):
                z.write(f, f.relative_to(run_dir))       # zip içinde: weights/best.pt, args.yaml, results.csv, ...
    print(f"İndirilecek: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("Eğitim bitmedi → zip yok. Sıradaki oturum: MODE='devam', PREV_OUTPUT=<önceki output>/runs/" + run_dir.name)